# SEER RCC 2010–2018 — Exploratory Data Analysis

Renal cell carcinoma (RCC) cohort from SEER (2010–2018).

**Libraries:** pandas, numpy, matplotlib, seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
DATA_PATH = "seer_rcc_2010_2018_clean.csv"

LABELS = {
    "sex": {0: "Female", 1: "Male"},
    "vital_status": {0: "Alive", 1: "Dead"},
    "metastasis": {0: "No", 1: "Yes"},
    "t_stage": {0: "T0/TX", 1: "T1", 2: "T2", 3: "T3", 4: "T4"},
    "n_stage": {0: "N0", 1: "N1"},
    "grade": {0: "Unknown", 1: "G1", 2: "G2", 3: "G3", 4: "G4"},
    "histology_enc": {0: "Clear cell", 1: "Papillary", 2: "Chromophobe", 3: "Other"},
}

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 1. Data Quality & Overview

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isnull().sum(),
    "missing_%": (df.isnull().mean() * 100).round(2),
    "unique": df.nunique(),
})
display(overview)
df.describe(include="all").T

## 2. Demographics — Age & Sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["age"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")
df["sex"].map(LABELS["sex"]).value_counts().plot.pie(autopct="%1.1f%%", ax=axes[1])
axes[1].set_ylabel("")
axes[1].set_title("Sex Distribution")
plt.tight_layout()
plt.show()

## 3. Tumor Characteristics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["tumor_size_cm"].clip(upper=20), bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Tumor Size (cm, clipped at 20)")
df["t_stage"].map(LABELS["t_stage"]).value_counts().plot(kind="bar", ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("T Stage")
df["grade"].map(LABELS["grade"]).value_counts().plot(kind="bar", ax=axes[1, 0], color="coral")
axes[1, 0].set_title("Grade")
df["histology_enc"].map(LABELS["histology_enc"]).value_counts().plot(kind="bar", ax=axes[1, 1], color="seagreen")
axes[1, 1].set_title("Histology")
plt.tight_layout()
plt.show()

## 4. Metastasis & Survival

In [ ]:
met_cols = ["metastasis", "lung_met", "bone_met", "liver_met", "brain_met"]
met_rates = pd.Series({c: df[c].mean() * 100 for c in met_cols})
met_rates.index = ["Any", "Lung", "Bone", "Liver", "Brain"]
met_rates.plot(kind="barh", figsize=(8, 4), color="indianred")
plt.xlabel("Prevalence (%)")
plt.title("Metastasis Site Prevalence")
plt.gca().invert_yaxis()
plt.show()

pd.crosstab(df["metastasis"].map(LABELS["metastasis"]),
            df["vital_status"].map(LABELS["vital_status"]),
            normalize="index").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["survival_months"], bins=40, kde=True, ax=axes[0])
axes[0].axvline(df["survival_months"].median(), color="red", ls="--")
axes[0].set_title("Survival Time (months)")
sns.boxplot(x=df["vital_status"].map(LABELS["vital_status"]), y=df["survival_months"], ax=axes[1])
axes[1].set_ylim(0, 180)
axes[1].set_title("Survival by Vital Status")
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop("year_diagnosis")
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Feature Correlations")
plt.tight_layout()
plt.show()